In [1]:
%load_ext autoreload
%autoreload 2

import sys
import os
import getpass
from pathlib import Path

project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(project_root))

if not os.environ.get('GROQ_API_KEY'):
    os.environ['GROQ_API_KEY'] = getpass.getpass("Enter Groq API Key: ")
if not os.environ.get('GOOGLE_API_KEY'):
    os.environ['GOOGLE_API_KEY'] = getpass.getpass("Enter Google API Key: ")

Enter Groq API Key:  ········
Enter Google API Key:  ········


In [ ]:
%pip install pylate

In [21]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_chroma import Chroma

embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")
dense_vectorstore = Chroma(
    embedding_function=embeddings,
    persist_directory="~/Codes/rag-lab/data/chroma_naive_gemini",
)

raw = dense_vectorstore.get(include=["documents"])
leaf_texts = raw["documents"]
print(f"{len(leaf_texts)} chunks pulled from the existing dense vectorstore")

576 chunks pulled from the existing dense vectorstore


In [22]:
import json

raw = dense_vectorstore.get(include=["documents"])
with open(project_root / "data" / "leaf_texts.json", "w") as f:
    json.dump(raw["documents"], f)
print(f"Dumped {len(raw['documents'])} chunks")

Dumped 576 chunks


In [23]:
# Execute PyLate index builder script
!~/my-jupyter-env/bin/python ~/Codes/rag-lab/rag_lab/retrieval/colbert_index_build.py

Encoding documents (bs=32): 100%|██████████████| 18/18 [01:43<00:00,  5.76s/it]
Indexed 576 chunks.█████████████████████████████████████████████████████████ 1/1
Index: /home/mohnish/Codes/rag-lab/data/colbert_index
id_to_text: /home/mohnish/Codes/rag-lab/data/id_to_text.json


In [24]:
import json
from rag_lab.retrieval.colbert_backend import ColBERTVectorStoreShim

with open(project_root / "data" / "id_to_text.json") as f:
    id_to_text = json.load(f)

colbert_vectorstore = ColBERTVectorStoreShim(
    index_folder=str(project_root / "data" / "colbert_index"),
    id_to_text=id_to_text,
)

Loading weights: 100%|█████████████████████████████████| 199/199 [00:00<00:00, 8458.74it/s]


In [25]:
# Raw PyLate result inspection smoke test
raw = colbert_vectorstore.retriever.retrieve(
    queries_embeddings=colbert_vectorstore._embed_query("test query"),
    k=2,
)
print("Raw PyLate result — inspect field names:")
print(raw)

Raw PyLate result — inspect field names:
[[{'id': 'chunk_14', 'score': 18.892578125}, {'id': 'chunk_13', 'score': 15.813720703125}]]


In [26]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0,
)

In [27]:
# Test vectorstore shim retrieval interface
retriever = colbert_vectorstore.as_retriever(search_kwargs={"k": 2})
docs = retriever.invoke("test query")

for i, doc in enumerate(docs):
    print(f"--- Document {i+1} (Score: {doc.metadata.get('score')}) ---")
    print(doc.page_content[:200])
    print()

--- Document 1 (Score: 18.892578125) ---
the observed fluctuations in the results may be attributed to these factors. These fluctuations may be
primarily due to the limited number of experiments performed and the non-optimized hyperparameter

--- Document 2 (Score: 15.813720703125) ---
4. Preliminary Results and Discussion
The results of our experiments, are summarized in Fig. 3 and will be discussed in the following: First,
we can observe that our proposed query strategy outperform



In [28]:
print(id_to_text["chunk_439"][:200])
print(id_to_text["chunk_993"][:200])

A PREPRINT - JULY 30, 2020
Algorithm 1: Adaptive weights for physics-informed neural networks
Consider a physics-informed neural networku(x,θ) with parametersθ and a loss function
L(θ) :=λbLb(θ) +λrLr


KeyError: 'chunk_993'

In [29]:
# How many chunks in id_to_text actually have unique text?
texts = list(id_to_text.values())
print(f"{len(texts)} total chunks, {len(set(texts))} unique")

576 total chunks, 576 unique


In [30]:
from collections import Counter

counts = Counter(leaf_texts)
most_common = counts.most_common(5)
for text, count in most_common:
    print(f"×{count}:", text[:100])

×1: Active Learning with Physics-Informed Graph Neural
Networks on Unstructured Meshes
Jens Decke1,∗, Al
×1: intensive, especially for high-dimensional problems and complex geometries. In recent years, Physics
×1: physics problems and seamlessly handle high-dimensional spaces, providing a flexible and efficient
I
×1: approach to solving complex PDEs. In Fig. 1 an active learning (AL) cycle with a PINN as Model is
de
×1: to compute spatial gradients [ 8]. In contrast, unstructured meshes provide the flexibility to handl


In [33]:
print(len(leaf_texts))
print(len(set(leaf_texts)))

576
576


In [32]:
print(dense_vectorstore._collection.count())

576
